In [1]:
# Importing the global libraries
import importlib, sys, os, pandas as pd
# from dotenv import load_dotenv
import pyspark.sql.types as T
import pyspark.sql as sql
import pyspark.sql.functions as F
import numpy as np
import datetime as dt
import random as rd
import subprocess
from xml.etree import ElementTree as ET

os.environ["JAVA_TOOL_OPTIONS"] = "-Djava.security.manager=allow"
os.environ["PYSPARK_SUBMIT_ARGS"] = "--conf spark.driver.extraJavaOptions=-Djava.security.manager=allow pyspark-shell"

#Mandatory
importlib.reload(importlib)
%load_ext autoreload
%autoreload 2

In [3]:
START_DATE : str = "2025-01-01"						# Simulation start date
START_TIME : str = "02:00:00"						# Simulation start time
NB_DAYS : int = 1									# Number of days to simulate
EDGES_MAX_SPEED : float = 27.78						# Max speed of the edges in m/s (120 km/h)
TRAIN_SPEED : float = 33.33							# Train speed in m/s (120 km/h)
DIRECTORY : str = "sumo_data"
OUTPUT_DIRECTORY : str = "new_start"
START_DATETIME : dt.datetime = dt.datetime.strptime(f"{START_DATE} {START_TIME}", "%Y-%m-%d %H:%M:%S")
END_DATETIME : dt.datetime = START_DATETIME + dt.timedelta(days=NB_DAYS)

filenames = {
	"punctuality" : f"{DIRECTORY}/punctuality/202501.csv",
	"trains" : f"{DIRECTORY}/trains.add.xml",
	"tracks_xml" : f"{OUTPUT_DIRECTORY}/tracks.edg.xml",
	"tracks_csv" : f"{DIRECTORY}/main_tracks.csv",
	"network" : f"{OUTPUT_DIRECTORY}/network.net.xml",
	"platforms_xml" : f"{OUTPUT_DIRECTORY}/platforms.add.xml",
	"platforms_csv" : "station_track_assigned.csv",
	"station_to_station" : f"{DIRECTORY}/station_to_station.csv",
	"schedule" : f"{OUTPUT_DIRECTORY}/schedule.trips.xml"
}

output = {
	"routes" : f"{OUTPUT_DIRECTORY}/routes.rou.xml",
	"schedule" : f"{OUTPUT_DIRECTORY}/schedule.trips.xml",
	"weight_src" : f"{OUTPUT_DIRECTORY}/weights.src.xml",
	"weight_dst" : f"{OUTPUT_DIRECTORY}/weights.dst.xml",
}

In [4]:
spark : sql.SparkSession = (sql.SparkSession.builder
	.appName("RailwaySimulationGenerator")
	.config("spark.driver.extraJavaOptions", "-Djava.security.manager=allow")
	.getOrCreate()
)

## Data extraction

### Tracks

In [6]:
tracks_df = spark.read.csv(filenames["tracks_csv"], header=True, inferSchema=True, sep=";")
tracks_dict = dict()
schema : T.DataType = T.ArrayType(T.ArrayType(T.DoubleType()))
tracks_df = tracks_df.withColumn(
	"Path",
	F.from_json(F.col("Path"), schema)
)
for row in tracks_df.collect() :
	track_id = int(row["ID"])
	start = row["Departure_switch"]
	end = row["Arrival_switch"]
	distance = round(row["Length_m"], 6) 
	if track_id not in tracks_dict :
		tracks_dict[track_id] = {}
	tracks_dict[track_id] = {
		"start" : start,
		"end" : end,
		"distance" : distance
	}

In [7]:
for i in range(5) :
	track_id = rd.choice(list(tracks_dict.keys()))
	print(track_id, tracks_dict[track_id])

2306 {'start': 2606, 'end': 2603, 'distance': 353.88437}
9569 {'start': 9835, 'end': 9880, 'distance': 125.213687}
10520 {'start': 10750, 'end': 10751, 'distance': 51.190244}
10041 {'start': 9527, 'end': 2297, 'distance': 7414.014822}
5486 {'start': 5836, 'end': 5837, 'distance': 93.259795}


In [8]:
tracks = ET.parse(filenames["tracks_xml"]).getroot()
edges = tracks.findall("edge")
edges_dict = dict()

for edge in edges:
	edge_id = int(edge.get("id"))
	from_node = int(edge.get("from"))
	to_node = int(edge.get("to"))
	for track in tracks_dict : 
		if (tracks_dict[track]["start"] == from_node and tracks_dict[track]["end"] == to_node) :
			edges_dict[edge_id] = {
				"track_id" : track,
				"from" : from_node,
				"to" : to_node,
				"length" : tracks_dict[track]["distance"]
			}
			break
		if tracks_dict[track]["start"] == to_node and tracks_dict[track]["end"] == from_node :
			edges_dict[edge_id] = {
				"track_id" : track,
				"from" : to_node,
				"to" : from_node,
				"length" : tracks_dict[track]["distance"]
			}
			break

In [9]:
for i in range(5) :
	edge_id = rd.choice(list(edges_dict.keys()))
	print(edge_id, edges_dict[edge_id])

35491 {'track_id': 17745, 'from': 16549, 'to': 3544, 'length': 29.041547}
23520 {'track_id': 11760, 'from': 11874, 'to': 11873, 'length': 35.537205}
34248 {'track_id': 17124, 'from': 16061, 'to': 13508, 'length': 44.918687}
5232 {'track_id': 2616, 'from': 2163, 'to': 1315, 'length': 411.474244}
10266 {'track_id': 5133, 'from': 5421, 'to': 5346, 'length': 15.545004}


### Station platforms 

In [10]:
plaforms_df = spark.read.csv(filenames["platforms_csv"], header=True, inferSchema=True, sep=",")
platforms_dict = dict()
for row in plaforms_df.collect() :
	station_id = int(row["Station_ID"])
	track_id = row["track_id"]
	city = row["Station_name"]
	if station_id not in platforms_dict :
		platforms_dict[station_id] = {
			"name" : city,
			"platforms" : []
		}
	platforms_dict[station_id]["platforms"].append(track_id)

In [11]:
for i in range(5) :
	platform = rd.choice(list(platforms_dict.keys()))
	print(platform, platforms_dict[platform])

924 {'name': 'OLEN', 'platforms': [14091, 14135]}
221 {'name': 'BRUXELLES-NORD', 'platforms': [23558, 23509, 23510, 23105, 23440, 23441, 23606, 23600, 23409, 23599, 23604, 23602]}
991 {'name': 'REMICOURT', 'platforms': [5369, 5411]}
1145 {'name': 'TIELT', 'platforms': [9353, 9665]}
743 {'name': 'LISSEWEGE', 'platforms': [2737, 2764]}


In [12]:
trainstop_dict = dict()
for station_id in platforms_dict :
	if station_id not in trainstop_dict :
		trainstop_dict[station_id] = {
			"name" : platforms_dict[station_id]["name"],
			"platforms" : []
		}
	for edge_id in edges_dict :
		track_id = edges_dict[edge_id]["track_id"]
		if track_id in platforms_dict[station_id]["platforms"] :
			trainstop_dict[station_id]["platforms"].append(int(edge_id))

In [13]:
for i in range(5) :
	station_id = rd.choice(list(trainstop_dict.keys()))
	print(station_id, trainstop_dict[station_id])

286 {'name': 'COURCELLES-MOTTE', 'platforms': [30070, 30071, 47394, 47395, 47396, 47397, 47732, 47733]}
130 {'name': 'BEERVELDE', 'platforms': [14438, 14439, 38060, 38061]}
1192 {'name': 'VILVOORDE', 'platforms': [8964, 8965, 9554, 9555, 46902, 46903, 46904, 46905, 46990, 46991, 49944, 49945]}
781 {'name': 'MALDEREN', 'platforms': [16624, 16625, 20018, 20019]}
939 {'name': 'OUDENAARDE', 'platforms': [3998, 3999, 4000, 4001, 4072, 4073, 4080, 4081, 4084, 4085]}


### Punctuality Data

In [14]:
punctuality_data_df = spark.read.csv(filenames["punctuality"], header=True, inferSchema=True, sep=";")
window : sql.Window = (
	sql.Window.partitionBy("TRAIN_NO","REAL_DATE_DEP") 
	.orderBy("PLANNED_DATETIME_DEP")
)
punctuality_data_df = (
	punctuality_data_df
	.withColumn(
		"NEXT_STOPPING_PLACE_ID",
		F.lead("STOPPING_PLACE_ID").over(window)
	)
)
punctuality_data_df = (punctuality_data_df.filter(
	(F.col("PLANNED_DATETIME_DEP") >= F.lit(START_DATETIME.strftime("%Y-%m-%d %H:%M:%S"))) & 
	(F.col("PLANNED_DATETIME_DEP") <= F.lit(END_DATETIME.strftime("%Y-%m-%d %H:%M:%S"))))
	.orderBy("TRAIN_NO", "REAL_DATE_DEP")
)

In [15]:
punctuality_data_df.show(5, truncate=False)

+--------+--------+----------+-----------------+-----------+---------+---------+----------------------------------------+-----------+-------------+-------------+--------------------+--------------------+----------------------+
|TRAIN_NO|RELATION|TRAIN_SERV|STOPPING_PLACE_ID|LINE_NO_DEP|DELAY_ARR|DELAY_DEP|RELATION_DIRECTION                      |LINE_NO_ARR|REAL_DATE_ARR|REAL_DATE_DEP|PLANNED_DATETIME_ARR|PLANNED_DATETIME_DEP|NEXT_STOPPING_PLACE_ID|
+--------+--------+----------+-----------------+-----------+---------+---------+----------------------------------------+-----------+-------------+-------------+--------------------+--------------------+----------------------+
|10      |ICE     |SNCB/NMBS |825              |37         |243      |243      |ICE: FRANKFURT(MAIN) HBF -> BRUSSEL-ZUID|37         |01-01-2025   |01-01-2025   |2025-01-01 20:28:00 |2025-01-01 20:28:00 |266                   |
|10      |ICE     |SNCB/NMBS |266              |37         |297      |297      |ICE: FRANKFU

### Station to station

In [16]:
station_to_station_df = spark.read.csv(filenames["station_to_station"], header=True, inferSchema=True, sep=";")
punctuality_data_df.show(5, truncate=False)

+--------+--------+----------+-----------------+-----------+---------+---------+----------------------------------------+-----------+-------------+-------------+--------------------+--------------------+----------------------+
|TRAIN_NO|RELATION|TRAIN_SERV|STOPPING_PLACE_ID|LINE_NO_DEP|DELAY_ARR|DELAY_DEP|RELATION_DIRECTION                      |LINE_NO_ARR|REAL_DATE_ARR|REAL_DATE_DEP|PLANNED_DATETIME_ARR|PLANNED_DATETIME_DEP|NEXT_STOPPING_PLACE_ID|
+--------+--------+----------+-----------------+-----------+---------+---------+----------------------------------------+-----------+-------------+-------------+--------------------+--------------------+----------------------+
|10      |ICE     |SNCB/NMBS |825              |37         |243      |243      |ICE: FRANKFURT(MAIN) HBF -> BRUSSEL-ZUID|37         |01-01-2025   |01-01-2025   |2025-01-01 20:28:00 |2025-01-01 20:28:00 |266                   |
|10      |ICE     |SNCB/NMBS |266              |37         |297      |297      |ICE: FRANKFU

In [17]:
network = dict()
for row in station_to_station_df.collect():
	station_a = row["Departure_station_id"]
	station_b = row["Arrival_station_id"]
	if station_a not in network:
		network[station_a] = []
	network[station_a].append(station_b)

In [18]:
for i in range(5) :
	station_id = rd.choice(list(network.keys()))
	print(station_id, network[station_id])

754 [148, 438]
132 [187, 9, 686]
810 [634, 325, 1663, 811, 871, 1224, 1048, 219, 1245]
649 [868, 160, 520, 1184, 604]
267 [266, 1159]


## Trip generation

### Retrieving trips from punctuality data

In [19]:
punctuality_data_df = punctuality_data_df.toPandas()

trips = {}

for train_id, group in punctuality_data_df.groupby("RELATION"):
	stops = group["STOPPING_PLACE_ID"].tolist()
	# next_stops = group["NEXT_STOPPING_PLACE_ID"].tolist()

	path = []
	for s in stops :
		if s in path :
			break
		path.append(s) 
	trips[train_id] = path

In [20]:
punctuality_data_df.groupby("TRAIN_NO").head(20)

,TRAIN_NO,RELATION,TRAIN_SERV,STOPPING_PLACE_ID,LINE_NO_DEP,DELAY_ARR,DELAY_DEP,RELATION_DIRECTION,LINE_NO_ARR,REAL_DATE_ARR,REAL_DATE_DEP,PLANNED_DATETIME_ARR,PLANNED_DATETIME_DEP,NEXT_STOPPING_PLACE_ID
0,10,ICE,SNCB/NMBS,825,37,243.0,243,ICE: FRANKFURT(MAIN) HBF -> BRUSSEL-ZUID,37,01-01-2025,01-01-2025,2025-01-01 20:28:00,2025-01-01 20:28:00,266.0
1,10,ICE,SNCB/NMBS,266,37,297.0,297,ICE: FRANKFURT(MAIN) HBF -> BRUSSEL-ZUID,3,01-01-2025,01-01-2025,2025-01-01 20:39:00,2025-01-01 20:39:00,27.0
2,10,ICE,SNCB/NMBS,27,37,276.0,276,ICE: FRANKFURT(MAIN) HBF -> BRUSSEL-ZUID,37,01-01-2025,01-01-2025,2025-01-01 20:40:00,2025-01-01 20:40:00,726.0
3,10,ICE,SNCB/NMBS,726,36,223.0,127,ICE: FRANKFURT(MAIN) HBF -> BRUSSEL-ZUID,37,01-01-2025,01-01-2025,2025-01-01 20:43:00,2025-01-01 20:46:00,31.0
4,10,ICE,SNCB/NMBS,31,2,62.0,62,ICE: FRANKFURT(MAIN) HBF -> BRUSSEL-ZUID,36,01-01-2025,01-01-2025,2025-01-01 20:51:00,2025-01-01 20:51:00,715.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42545,19972,IC 19-2,SNCB/NMBS,1154,94,NaN,17,IC 19-2: TOURNAI -> LILLE FLANDRES,None,None,01-01-2025,NaT,2025-01-01 20:36:00,427.0
42546,19972,IC 19-2,SNCB/NMBS,427,94,34.0,40,IC 19-2: TOURNAI -> LILLE FLANDRES,94,01-01-2025,01-01-2025,2025-01-01 20:40:00,2025-01-01 20:40:00,NaN
42547,19973,IC 19-2,SNCB/NMBS,427,94,-24.0,30,IC 19-2: LILLE FLANDRES -> TOURNAI,94,01-01-2025,01-01-2025,2025-01-01 22:16:00,2025-01-01 22:17:00,NaN
42548,19976,IC 19-2,SNCB/NMBS,1154,94,NaN,49,IC 19-2: TOURNAI -> LILLE FLANDRES,None,None,01-01-2025,NaT,2025-01-01 21:44:00,427.0


In [21]:
final_trips  = {}
for trip in trips :
	path = []
	for s in trips[trip] :
		path.append(s) if s in platforms_dict else None
	if len(path) > 1 :
		final_trips[trip] = path

In [22]:
for train_id in final_trips :
	path = []
	for s in final_trips[train_id] :
		path.append((s, platforms_dict[s]["name"], s in platforms_dict))
	print(train_id, path)

EURST [(504, 'HALLE', True), (220, 'BRUXELLES-MIDI', True), (217, 'BRUXELLES-CHAPELLE', True), (215, 'BRUXELLES-CENTRAL', True), (216, 'BRUXELLES-CONGRES', True), (221, 'BRUXELLES-NORD', True), (1048, 'SCHAERBEEK', True), (810, 'MECHELEN', True), (811, 'MECHELEN-NEKKERSPOEL', True), (1083, 'SINT-KATELIJNE-WAVER', True), (336, 'DUFFEL', True), (644, 'KONTICH-LINT', True), (590, 'HOVE', True), (866, 'MORTSEL-OUDE GOD', True), (863, 'MORTSEL', True), (139, 'ANTWERPEN-BERCHEM', True), (37, 'ANTWERPEN-CENTRAAL', True), (764, 'ANTWERPEN-LUCHTBAL', True), (1839, 'NOORDERKEMPEN', True)]
EXTRA [(259, 'CHARLEROI-CENTRAL', True), (791, 'MARCHIENNE-AU-PONT', True), (1018, 'ROUX', True), (286, 'COURCELLES-MOTTE', True), (768, 'LUTTRE', True), (919, 'OBAIX-BUZET', True), (523, 'HASSELT', True), (636, 'KIEWIT', True), (184, 'BOKRIJK', True), (621, 'JETTE', True), (1767, 'BOCKSTAEL', True), (221, 'BRUXELLES-NORD', True), (216, 'BRUXELLES-CONGRES', True), (215, 'BRUXELLES-CENTRAL', True), (217, 'BRUXEL

In [23]:
starting_stations = set()
for train_id in final_trips :
	path = final_trips[train_id]
	if len(path) > 0 :
		starting_stations.add(path[0])
		starting_stations.add(path[-1])

for station in starting_stations :
	print(station, trainstop_dict[station]["name"])

520 HARELBEKE
523 HASSELT
19 ALKEN
31 ANS
37 ANTWERPEN-CENTRAAL
1061 SCHULEN
1063 SCLESSIN
553 HERENT
2089 ANDERLECHT
1067 SERAING
567 HEVERLEE
1085 LIERDE
1088 SINT-NIKLAAS
70 ARLON
602 IEPER
604 INGELMUNSTER
615 JEMAPPES
1128 TEMSE
617 ROCHEFORT-JEMELLE
1154 TOURNAI
130 BEERVELDE
1161 TURNHOUT
139 ANTWERPEN-BERCHEM
146 BERTRIX
158 BINCHE
673 LA LOUVIERE-CENTRE
1195 VISE
684 LANDEN
1198 VIVILLE
1206 WAARSCHOOT
1730 SINT-GILLIS(DENDERMONDE)
1224 WEERDE
201 BRACQUEGNIES
1226 WELKENRAEDT
1744 LA LOUVIERE-SUD
210 BRUGGE
723 LIBRAMONT
1238 WEZEMAAL
726 LIEGE-GUILLEMINS
728 LIEGE-CARRE
216 BRUXELLES-CONGRES
1242 WIJGMAAL
218 BRUXELLES-LUXEMBOURG
220 BRUXELLES-MIDI
221 BRUXELLES-NORD
733 LIERS
223 BRUXELLES-OUEST
1260 ZAVENTEM
1265 ZELE
1274 ZOTTEGEM
258 CHARLEROI-OUEST
259 CHARLEROI-CENTRAL
272 CINEY
788 MARCHE-EN-FAMENNE
791 MARCHIENNE-AU-PONT
798 MARIEMBOURG
801 MARLOIE
810 MECHELEN
1839 NOORDERKEMPEN
1842 MESSANCY
825 HERGENRATH
318 DENDERLEEUW
319 DENDERMONDE
320 DE PINTE
1350 ZONHOVEN


### Method 1 : creating trips manually

#### Creating network as a graph

In [24]:
temp = {}
for edge_id in edges_dict:
	start, end = edges_dict[edge_id]["from"], edges_dict[edge_id]["to"]
	if start not in temp :
		temp[start] = set()
	if end not in temp :
		temp[end] = set()
	temp[start].add(edge_id)
	temp[end].add(edge_id)

In [25]:
import networkx as nx

G = nx.Graph()

for node in temp :
	tracks = list(temp[node])
	for i in range(len(tracks)) :
		for j in range(len(tracks)) : 
			if i != j :
				a = tracks[i]
				b = tracks[j]
				G.add_edge(a, b, track_id=f"{a} - {b}", weight=edges_dict[a]["length"])

#### Creating paths between stations

In [29]:
def find_path_between_stations(G, station_A, station_B, station_A_tracks, station_B_tracks, station_to_station_paths):
	if (station_A, station_B) not in station_to_station_paths :
		station_to_station_paths[(station_A, station_B)] = dict()
	# print(station_to_station_paths[(station_A, station_B)])
		for platform_A in station_A_tracks :
			for platform_B in station_B_tracks :
				if (platform_A, platform_B) not in station_to_station_paths[(station_A, station_B)] :
					try:
						path = nx.shortest_path(G, platform_A, platform_B, weight="weight")
						if len(path) == 0 :
							path = None
						else :
							length = nx.path_weight(G, path, weight="weight")
						if path is not None :
							station_to_station_paths[(station_A, station_B)][(platform_A, platform_B)] = (path, length)
						else :
							station_to_station_paths[(station_A, station_B)][(platform_A, platform_B)] = None
					except Exception as e:
						station_to_station_paths[(station_A, station_B)][(platform_A, platform_B)] = None
						# continue

In [30]:
station_to_station_paths = dict()
for station_id in network :
	for neighbor in network[station_id] :
		if station_id in trainstop_dict and neighbor in trainstop_dict :
			station_id = int(station_id)
			neighbor = int(neighbor)
			print(f"Finding path between {trainstop_dict[station_id]['name']} and {trainstop_dict[neighbor]['name']}")
			tracks_A = trainstop_dict[station_id]["platforms"]
			tracks_B = trainstop_dict[neighbor]["platforms"]
			find_path_between_stations(G, station_id, neighbor, tracks_A, tracks_B, station_to_station_paths)
			find_path_between_stations(G, neighbor, station_id, tracks_B, tracks_A, station_to_station_paths)

Finding path between MOLLEM and MERCHTEM
Finding path between MOLLEM and ASSE
Finding path between CALLENELLE and PERUWELZ
Finding path between CALLENELLE and MAUBRAY
Finding path between GENVAL and LA HULPE
Finding path between GENVAL and RIXENSART
Finding path between MAFFLE and ATH
Finding path between MAFFLE and MEVERGNIES-ATTRE
Finding path between STATTE and HUY
Finding path between STATTE and BAS-OHA
Finding path between BILZEN and TONGEREN
Finding path between BILZEN and BOKRIJK
Finding path between BILZEN and DIEPENBEEK
Finding path between ARCADES and DELTA
Finding path between ARCADES and BOONDAEL
Finding path between ARCADES and ETTERBEEK
Finding path between BEVERLO and LEOPOLDSBURG
Finding path between BEVERLO and BERINGEN
Finding path between NESSONVAUX and FRAIPONT
Finding path between NESSONVAUX and PEPINSTER
Finding path between DELTA and ARCADES
Finding path between DELTA and ETTERBEEK
Finding path between DELTA and MERODE
Finding path between DELTA and WATERMAEL
Fin

In [31]:
for trip in final_trips :
	path = final_trips[trip]
	for i in range(len(path) - 1):
		station_A = int(path[i])
		station_B = int(path[i + 1])
		if (station_A, station_B) not in station_to_station_paths :
			print(f"Finding path between {trainstop_dict[station_A]['name']} and {trainstop_dict[station_B]['name']}")
			tracks_A = trainstop_dict[station_A]["platforms"]
			tracks_B = trainstop_dict[station_B]["platforms"]
			find_path_between_stations(G, station_A, station_B, tracks_A, tracks_B, station_to_station_paths)
		if (station_B, station_A) not in station_to_station_paths : 
			tracks_A = trainstop_dict[station_B]["platforms"]
			tracks_B = trainstop_dict[station_A]["platforms"]
			find_path_between_stations(G, station_B, station_A, tracks_A, tracks_B, station_to_station_paths)

Finding path between MORTSEL-OUDE GOD and MORTSEL
Finding path between OBAIX-BUZET and HASSELT
Finding path between BOKRIJK and JETTE
Finding path between BRUXELLES-CHAPELLE and BRUGGE
Finding path between OOSTENDE and LIEGE-GUILLEMINS
Finding path between DOLHAIN-GILEPPE and LIEGE-SAINT-LAMBERT
Finding path between WELKENRAEDT and OOSTENDE
Finding path between HARELBEKE and BISSEGEM
Finding path between SCHAERBEEK and MONS
Finding path between ASSESSE and NATOYE
Finding path between YVOIR and BRUXELLES-MIDI
Finding path between HERENT and KNOKKE
Finding path between SCHAERBEEK and GENT-SINT-PIETERS
Finding path between ZAVENTEM and VICHTE
Finding path between MILMORT and MONS
Finding path between JEMEPPE-SUR-SAMBRE and MOUSTIER
Finding path between LINKEBEEK and MOENSBERG
Finding path between MOENSBERG and UCCLE-CALEVOET
Finding path between GOUVY and LIERS
Finding path between KAPELLE-OP-DEN-BOS and GENT-SINT-PIETERS
Finding path between POIX-SAINT-HUBERT and MARLOIE
Finding path bet

#### Retrieve the schedule from punctuality data

In [32]:
import pandas as pd

# def time_to_sumo_seconds(dt, simulation_start=None):
# 	dt = pd.to_datetime(dt, errors="coerce")
# 	if simulation_start is None:
# 		# secondes depuis minuit
# 		return dt.dt.day * 24 + dt.dt.hour * 3600 + dt.dt.minute * 60 + dt.dt.second
# 	else:
# 		# secondes depuis le début de la simulation
# 		simulation_start = pd.to_datetime(simulation_start)
# 		return (dt - simulation_start).dt.total_seconds()

def build_trips_from_punctuality(df):
	df = df.copy()

	# Nettoyage
	df = df.dropna(subset=[
		"TRAIN_NO",
		"STOPPING_PLACE_ID",
		# "PLANNED_DATETIME_DEP"
	])

	df["TRAIN_NO"] = df["TRAIN_NO"].astype(str)
	df["STOPPING_PLACE_ID"] = df["STOPPING_PLACE_ID"].astype(str)

	# Conversion datetime
	df["dep_datetime"] = pd.to_datetime(
		df["PLANNED_DATETIME_DEP"],
		errors="coerce"
	)

	df = df.dropna(subset=["dep_datetime"])

	# Secondes SUMO depuis minuit
	df["dep_seconds"] = (
		df["dep_datetime"].dt.day * 24 * 3600 
		+ df["dep_datetime"].dt.hour * 3600
		+ df["dep_datetime"].dt.minute * 60
		+ df["dep_datetime"].dt.second
	)

	# Tri important
	df = df.sort_values(["TRAIN_NO", "dep_datetime"])

	trips = []

	# for train_no, group in df.groupby("TRAIN_NO", "PLANNED_DATETIME_DEP"):
	cmpt = 1
	for (train_no, dep), group in df.groupby(["TRAIN_NO", "REAL_DATE_DEP"]):
		stations = group["STOPPING_PLACE_ID"].tolist()

		departure_times = dict(
			zip(group["STOPPING_PLACE_ID"], group["dep_seconds"])
		)

		trip = {
			"train_id": cmpt,
			"departure_date": dep,
			"stations": [int(s) for s in stations],
			"departure_times": {int(s) : float(departure_times[s]) for s in stations}
		}
		cmpt += 1

		trips.append(trip)

	return trips

In [33]:
scheduled_trips = build_trips_from_punctuality(punctuality_data_df)

In [34]:
for trip in scheduled_trips :
	for station in trip["departure_times"] :
		trip["departure_times"][station] = trip["departure_times"][station] - (START_DATETIME.hour * 3600 + START_DATETIME.minute * 60 + START_DATETIME.second)

In [35]:
print(f"Total number of trips : {len(scheduled_trips)}\n")
# for i in range(5) :
# 	trip = rd.choice(scheduled_trips)
# 	print(f"trip id : {trip['train_id']}\nstations : {[(s,trainstop_dict[int(s)]['name']) if int(s) in trainstop_dict else None for s in trip['departure_times']]}\ndeparture times : {trip['departure_times']}\n")

for trip in scheduled_trips :
    print(f"trip id : {trip['train_id']}\ndate:{trip['departure_date']}\nstations : {[(s,trainstop_dict[int(s)]['name']) if int(s) in trainstop_dict else None for s in trip['departure_times']]}\ndeparture times : {trip['departure_times']}\n")

Total number of trips : 2403

trip id : 1
date:01-01-2025
stations : [(825, 'HERGENRATH'), (266, 'CHENEE'), (27, 'ANGLEUR'), (726, 'LIEGE-GUILLEMINS'), (31, 'ANS'), (715, 'LEUVEN'), (553, 'HERENT'), (1174, 'VELTEM'), (368, 'ERPS-KWERPS'), (648, 'KORTENBERG'), (916, 'NOSSEGEM'), (1260, 'ZAVENTEM'), (325, 'DIEGEM'), (521, 'HAREN-SUD'), (1048, 'SCHAERBEEK'), (221, 'BRUXELLES-NORD'), (216, 'BRUXELLES-CONGRES'), (215, 'BRUXELLES-CENTRAL'), (217, 'BRUXELLES-CHAPELLE')]
departure times : {825: 152880.0, 266: 153540.0, 27: 153600.0, 726: 153960.0, 31: 154260.0, 715: 155460.0, 553: 155580.0, 1174: 155700.0, 368: 155760.0, 648: 155820.0, 916: 155880.0, 1260: 155940.0, 325: 156000.0, 521: 156060.0, 1048: 156180.0, 221: 156480.0, 216: 156600.0, 215: 156660.0, 217: 156780.0}

trip id : 2
date:01-01-2025
stations : [(220, 'BRUXELLES-MIDI'), (217, 'BRUXELLES-CHAPELLE'), (215, 'BRUXELLES-CENTRAL'), (216, 'BRUXELLES-CONGRES'), (221, 'BRUXELLES-NORD'), (1048, 'SCHAERBEEK'), (521, 'HAREN-SUD'), (325, 'DI

#### Creating the schedule

In [114]:
from collections import defaultdict
import heapq


def intervals_overlap(a_start, a_end, b_start, b_end, buffer_time=0):
	"""
	Vérifie si deux occupations de quai se chevauchent.
	buffer_time ajoute une marge de sécurité avant/après.
	"""
	return a_start < b_end + buffer_time and b_start < a_end + buffer_time


def is_platform_available(platform_id, arrival, departure, platform_schedule, buffer_time=60):
	"""
	Vérifie si un quai est libre entre arrival et departure.
	"""
	for reserved in platform_schedule[platform_id]:
		if intervals_overlap(
			arrival,
			departure,
			reserved["arrival"],
			reserved["departure"],
			buffer_time
		):
			return False

	return True


def reserve_platform(platform_id, train_id, station, arrival, departure, platform_schedule):
	"""
	Réserve un quai pour un train.
	"""
	platform_schedule[platform_id].append({
		"train_id": train_id,
		"station": station,
		"arrival": arrival,
		"departure": departure
	})


def estimate_travel_time(length_m, average_speed_mps=33.33):
	"""
	Estime le temps de trajet.
	"""
	return length_m / average_speed_mps


# def choose_best_platform_pair(
#     station_A,
#     station_B,
#     candidate_platforms_A,
#     candidate_platforms_B,
#     station_to_station_paths
# ):
#     """
#     Choisit la paire de quais qui possède un chemin faisable entre deux stations.
#     On prend ici le chemin le plus court.
#     """
#     best = None

#     possible_paths = station_to_station_paths.get((station_A, station_B), {})

#     for platform_A in candidate_platforms_A:
#         for platform_B in candidate_platforms_B:
#             key = (platform_A, platform_B)

#             if key not in possible_paths:
#                 continue

#             path, length = possible_paths[key]

#             if best is None or length < best["length"]:
#                 best = {
#                     "platform_A": platform_A,
#                     "platform_B": platform_B,
#                     "path": path,
#                     "length": length
#                 }

#     return best

In [158]:
def assign_trip_with_fixed_times(
	trip,
	station_to_platforms,
	station_to_station_paths,
	platform_schedule,
	dwell_time=60,
	buffer_time=60,
	max_delay=0   # optionnel
):
	train_id = trip["train_id"]
	stations = []
	for s in trip["stations"] :
		if s is not None and s in station_to_platforms :
			stations.append(s)	
	dep_times = {s : trip["departure_times"][s] for s in stations}
	# print(stations)
	# print(dep_times)

	assigned_stops = []
	full_path = []

	previous_platform = None

	for i, station in enumerate(stations):

		departure = dep_times[station]
		arrival = departure - dwell_time

		candidate_platforms = station_to_platforms.get(station, [])

		if not candidate_platforms:
			print(f"Aucun quai pour {station_to_platforms[station]['name']}")
			return None, f"Aucun quai pour {station_to_platforms[station]['name']}"

		# PREMIÈRE STATION
		if i == 0:
			chosen = None

			for p in candidate_platforms:
				if is_platform_available(p, arrival, departure, platform_schedule, buffer_time):
					chosen = p
					break

			if chosen is None:
				print(f"Conflit quai départ {station_to_platforms[station]['name']}")
				return None, f"Conflit quai départ {station_to_platforms[station]['name']}"

			reserve_platform(chosen, train_id, station, arrival, departure, platform_schedule)

			assigned_stops.append({
				"station": station,
				"platform": chosen,
				"arrival": arrival,
				"departure": departure
			})

			previous_platform = chosen

		# STATIONS SUIVANTES
		else:
			prev_station = stations[i - 1]

			best_candidate = None

			for p in candidate_platforms:

				path_info = station_to_station_paths.get(
					(prev_station, station), {}
				).get((previous_platform, p))

				if path_info is None:
					print(f"No path info for {station_to_platforms[prev_station]['name']} → {station_to_platforms[station]['name']} via platforms {previous_platform} → {p}")
					continue
				else : 
					print(f"Path info for {station_to_platforms[prev_station]['name']} → {station_to_platforms[station]['name']} via platforms {previous_platform} → {p} : {path_info}")

				path, length = path_info
				travel_time = estimate_travel_time(length)

				# ⚠️ CONTRAINTE TEMPORELLE CRITIQUE
				prev_departure = dep_times[prev_station]
				expected_arrival = prev_departure + travel_time

				# ❌ impossible physiquement
				if expected_arrival > arrival:
					continue

				# check quai libre
				if not is_platform_available(p, arrival, departure, platform_schedule, buffer_time):
					continue

				candidate = {
					"platform": p,
					"path": path,
					"length": length
				}

				# if best_candidate is None or length < best_candidate["length"]:
				if best_candidate is None :
					best_candidate = candidate
					break

			if best_candidate is None:
				print(f"Infeasible: {station_to_platforms[prev_station]['name']} → {station_to_platforms[station]['name']}")
				return None, f"Infeasible: {station_to_platforms[prev_station]['name']} → {station_to_platforms[station]['name']}"

			reserve_platform(
				best_candidate["platform"],
				train_id,
				station,
				arrival,
				departure,
				platform_schedule
			)

			assigned_stops.append({
				"station": station,
				"platform": best_candidate["platform"],
				"arrival": arrival,
				"departure": departure
			})

			full_path.extend(best_candidate["path"])
			previous_platform = best_candidate["platform"]

	return {
		"train_id": train_id,
		"stops": assigned_stops,
		"edges": full_path
	}, None

In [155]:
def generate_feasible_timetable(
	trips,
	station_to_platforms,
	station_to_station_paths,
	dwell_time=60,
	buffer_time=60,
):
	platform_schedule = defaultdict(list)

	feasible_trips = []
	rejected_trips = []

	# Important : traiter les trains dans l'ordre chronologique
	trips = sorted(trips, key=lambda x: x["departure_times"][x["stations"][0]])

	for trip in trips:
		if len(trip["stations"]) >= 2 :
			result, error = assign_trip_with_fixed_times(
				trip=trip,
				station_to_platforms=station_to_platforms,
				station_to_station_paths=station_to_station_paths,
				platform_schedule=platform_schedule,
				dwell_time=dwell_time,
				buffer_time=buffer_time,
			)

			if error:
				rejected_trips.append({
					"train_id": trip["train_id"],
					"reason": error
				})
			else:
				feasible_trips.append(result)

	return feasible_trips, rejected_trips, platform_schedule

In [165]:
# for pair in station_to_station_paths :
#     print(f"Station pair : {pair}")
#     for platforms in station_to_station_paths[pair] :
#         print(f"  Platforms : {platforms} → {station_to_station_paths[pair][platforms]}")
print(station_to_station_paths[(1530, 218)])

KeyError: (1530, 218)

In [159]:
feasible_trips, rejected_trips, platform_schedule = generate_feasible_timetable(
	trips=scheduled_trips,
	station_to_platforms=trainstop_dict,
	station_to_station_paths=station_to_station_paths,
	dwell_time=60,
	buffer_time=60
)


No path info for AALST → EREMBODEGEM via platforms name → name
No path info for AALST → EREMBODEGEM via platforms name → platforms
Infeasible: AALST → EREMBODEGEM
No path info for ETTERBEEK → BRUXELLES-LUXEMBOURG via platforms platforms → name
No path info for ETTERBEEK → BRUXELLES-LUXEMBOURG via platforms platforms → platforms
Infeasible: ETTERBEEK → BRUXELLES-LUXEMBOURG
Conflit quai départ GEMBLOUX
Conflit quai départ DUFFEL
Conflit quai départ DUFFEL
Conflit quai départ BRUXELLES-MIDI
Conflit quai départ ZICHEM
Conflit quai départ HILLEGEM
Conflit quai départ SINT-NIKLAAS
Conflit quai départ BRUXELLES-NORD
Conflit quai départ DENDERLEEUW
Conflit quai départ BRUXELLES-CHAPELLE
Conflit quai départ WATERLOO
Conflit quai départ RHODE-SAINT-GENESE
Conflit quai départ BRUXELLES-CENTRAL
Conflit quai départ OOSTKAMP
Conflit quai départ ATHUS
No path info for HASSELT → ALKEN via platforms name → name
No path info for HASSELT → ALKEN via platforms name → platforms
Infeasible: HASSELT → ALKEN


KeyboardInterrupt: 

In [152]:
print(f"Nombre de trains planifiés : {len(feasible_trips)}")
print(f"Nombre de trains rejetés : {len(rejected_trips)}")

Nombre de trains planifiés : 134
Nombre de trains rejetés : 13715


In [153]:
for trip in feasible_trips :
	# print(f"Train {trip['train_id']} :")
	# for stop in trip["stops"] :
	# 	station_name = trainstop_dict[stop['station']]["name"] if stop['station'] in trainstop_dict else None
	# 	print(f"\tStation {stop['station']} ({station_name}), Quai {stop['platform']}, Arrivée {stop['arrival']}s, Départ {stop['departure']}s")
	print(trip)

{'train_id': 1716, 'stops': [{'station': 868, 'platform': 'name', 'arrival': 109740.0, 'departure': 109800.0}], 'edges': []}
{'train_id': 1729, 'stops': [{'station': 868, 'platform': 'name', 'arrival': 113340.0, 'departure': 113400.0}], 'edges': []}
{'train_id': 1736, 'stops': [{'station': 868, 'platform': 'name', 'arrival': 116940.0, 'departure': 117000.0}], 'edges': []}
{'train_id': 1719, 'stops': [{'station': 868, 'platform': 'platforms', 'arrival': 117300.0, 'departure': 117360.0}], 'edges': []}
{'train_id': 1743, 'stops': [{'station': 868, 'platform': 'name', 'arrival': 120540.0, 'departure': 120600.0}], 'edges': []}
{'train_id': 1817, 'stops': [{'station': 868, 'platform': 'name', 'arrival': 124140.0, 'departure': 124200.0}], 'edges': []}
{'train_id': 1752, 'stops': [{'station': 868, 'platform': 'platforms', 'arrival': 124500.0, 'departure': 124560.0}], 'edges': []}
{'train_id': 1812, 'stops': [{'station': 868, 'platform': 'name', 'arrival': 127740.0, 'departure': 127800.0}], 'ed

### Method 2 : Parsing auto-generated trips from randomTrips.py

#### Weight station

In [ ]:
weight_stations_str = (
	'<?xml version="1.0" encoding="UTF-8"?>\n' + 
	'<edgedata>\n' +
	f'\t<interval begin="0" end="{1 * 24 * 3600}">\n'
)

for station_id in starting_stations :
	for track_id in trainstop_dict[station_id]["platforms"] :
		weight_stations_str += f'\t\t<edge id="{track_id}" value="1">\n'

weight_stations_str += (
	'\t</interval>\n' + 
	'</edgedata>'
)
print(weight_stations_str)

<?xml version="1.0" encoding="UTF-8"?>
<edgedata>
	<interval begin="0" end="86400">
		<edge id="21656" value="1">
		<edge id="21657" value="1">
		<edge id="21754" value="1">
		<edge id="21755" value="1">
		<edge id="5522" value="1">
		<edge id="5523" value="1">
		<edge id="5524" value="1">
		<edge id="5525" value="1">
		<edge id="7276" value="1">
		<edge id="7277" value="1">
		<edge id="7278" value="1">
		<edge id="7279" value="1">
		<edge id="7348" value="1">
		<edge id="7349" value="1">
		<edge id="7350" value="1">
		<edge id="7351" value="1">
		<edge id="46024" value="1">
		<edge id="46025" value="1">
		<edge id="46974" value="1">
		<edge id="46975" value="1">
		<edge id="47034" value="1">
		<edge id="47035" value="1">
		<edge id="49450" value="1">
		<edge id="49451" value="1">
		<edge id="49520" value="1">
		<edge id="49521" value="1">
		<edge id="52006" value="1">
		<edge id="52007" value="1">
		<edge id="52010" value="1">
		<edge id="52011" value="1">
		<edge id="15780" value="1"

In [ ]:
with open(output["weight_src"], 'w', encoding = "utf-8") as f :
	f.write(weight_stations_str)

In [ ]:
with open(output["weight_dst"], 'w', encoding = "utf-8") as f :
	f.write(weight_stations_str)

#### RandomTrips.py

In [ ]:
sumo_home = os.environ.get("SUMO_HOME")
if not sumo_home:
	raise RuntimeError("Environment variable SUMO_HOME is not defined.")
tools = os.path.join(sumo_home, "tools")
tool_path = os.path.join(tools, "randomTrips.py")
if not os.path.isfile(tool_path):
	raise RuntimeError(f"randomTrips.py not found in {tools}")

cmd = [
	"py",								# ensure Python is available
	f'"{tool_path}"',
	"-n", f"{OUTPUT_DIRECTORY}/network.net.xml",				# input: the compiled SUMO network
	"-o", f"{OUTPUT_DIRECTORY}/schedule.trips.xml",			# output: generated trips file
	"-a", f"sumo_data/trains.add.xml",				# use the defined vehicle type(s)
	"--trip-attributes", 'type="myTrain"',				# assign the vType 'myTrain' to each trip
	"--edge-permission", "rail",						# ensure only rail edges are used
	"-b", str(0),				# begin time for trip generation
	"-e", str(86400),			# end time for trip generation
	"--insertion-rate", "200,600,1100,700,500,900,1200,400",		# train insertion rate (s)	
	"--min-distance", "1000",		# minimum origin-destination distance
	"--weights-prefix", f"{OUTPUT_DIRECTORY}/weights"
]
print(" ".join(cmd))

py "C:\Program Files (x86)\Eclipse\Sumo\tools\randomTrips.py" -n new_start/network.net.xml -o new_start/schedule.trips.xml -a sumo_data/trains.add.xml --trip-attributes type="myTrain" --edge-permission rail -b 0 -e 86400 --insertion-rate 200,600,1100,700,500,900,1200,400 --min-distance 1000 --weights-prefix new_start/weights


In [ ]:
schedule_xml = ET.parse(filenames["schedule"]).getroot()
trips_xml = schedule_xml.findall("trip")

schedule_str = (
	'<?xml version="1.0" encoding="UTF-8"?>\n' + 
	'<routes>\n'
)

cmpt = 0
for trip in trips_xml :
	trip_number = trip.get("id")
	start, end = int(trip.get("from")), int(trip.get("to"))
	depart_time = trip.get("depart")
	start_station = None
	end_station = None
	for station_id in starting_stations :
		# print(start, end, trainstop_dict[station_id]["platforms"])
		if start in trainstop_dict[station_id]["platforms"] :
			start_station = station_id
		if end in trainstop_dict[station_id]["platforms"] :
			end_station = station_id
		if start is None and end is not None :
			break
	# print(trip_id, start, end, start_station, end_station)

	if start_station is not None and end_station is not None :
		# print(trip_id, start, end, trainstop_dict[start_station]["name"], trainstop_dict[end_station]["name"])
		for trip_id in final_trips : 
			path = final_trips[trip_id]
			if start_station in path and end_station in path :
				schedule_str += f'\t<trip id="{cmpt}" depart="{depart_time}" from="{start_station}" to="{end_station}" type="myTrain"/>\n'
				cmpt += 1
				break
schedule_str += '</routes>'

print(schedule_str)

<?xml version="1.0" encoding="UTF-8"?>
<routes>
	<trip id="0" depart="1098.00" from="220" to="365" type="myTrain"/>
	<trip id="1" depart="1782.00" from="220" to="1048" type="myTrain"/>
	<trip id="2" depart="3150.00" from="210" to="31" type="myTrain"/>
	<trip id="3" depart="3240.00" from="220" to="1048" type="myTrain"/>
	<trip id="4" depart="4158.00" from="205" to="553" type="myTrain"/>
	<trip id="5" depart="4788.00" from="210" to="929" type="myTrain"/>
	<trip id="6" depart="5130.00" from="553" to="218" type="myTrain"/>
	<trip id="7" depart="5994.00" from="1048" to="220" type="myTrain"/>
	<trip id="8" depart="6318.00" from="220" to="936" type="myTrain"/>
	<trip id="9" depart="7452.00" from="871" to="218" type="myTrain"/>
	<trip id="10" depart="9342.00" from="220" to="936" type="myTrain"/>
	<trip id="11" depart="9432.00" from="365" to="220" type="myTrain"/>
	<trip id="12" depart="11058.00" from="220" to="936" type="myTrain"/>
	<trip id="13" depart="11178.00" from="220" to="1048" type="my

In [ ]:
with open(output["schedule"], 'w', encoding = "utf-8") as f :
	f.write(schedule_str)

#### Generating routes for generated trips

In [ ]:
import shutil

path = shutil.which("duarouter")

cmd = [
	f"\"{path}\"",						# ensure duarouter is available
	"--net-file", f"{OUTPUT_DIRECTORY}/network.net.xml",			# input: compiled network
	"--route-files", f"{OUTPUT_DIRECTORY}/schedule.trips.xml",		# input: generated trips file
	"--additional-files", f"sumo_data/trains.add.xml",			# input: additional file with vehicle types
	"--output-file", f"{OUTPUT_DIRECTORY}/routes.rou.xml",			# output: generated routes file
	"--ignore-errors", "true",						# stop if any routing error occurs
]
print(" ".join(cmd))

"C:\Program Files (x86)\Eclipse\Sumo\bin\duarouter.EXE" --net-file new_start/network.net.xml --route-files new_start/schedule.trips.xml --additional-files sumo_data/trains.add.xml --output-file new_start/routes.rou.xml --ignore-errors true
